In [10]:
import os , json
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from datasets import Dataset

In [11]:
SFT_DATA_PATH = r"C:\datasets\llm-data\sft\sft_round1.json"

In [12]:
with open(SFT_DATA_PATH) as f:
    data = json.load(f)

print(len(data))

259


In [19]:
def format_prompt(sample):
    instruction = sample["instruction"].strip()
    input_text  = sample["input"].strip()
    output_text = sample["output"].strip()

    if input_text:
        text = f"""### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output_text}"""
    else:
        text = f"""### Instruction:\n{instruction}\n\n### Response:\n{output_text}"""

    return {"text": text}


In [20]:
formatted = [format_prompt(s) for s in data]
dataset = Dataset.from_list(formatted)
split = dataset.train_test_split(test_size=0.1, seed=42)

print(f"Train: {len(split['train'])} | Val: {len(split['test'])}")
print(f"\nSample:\n{split['train'][0]['text'][:300]}")

Train: 233 | Val: 26

Sample:
### Instruction:
How does matrix multiplication work in NumPy and what are the computational complexity considerations when multiplying large matrices?

### Response:
### Reasoning:
Matrix multiplication follows the rule that for matrices A(m×n) and B(n×p), the result C(m×p) where C[i,j] = Σ(A[i,k] 


In [ ]:
lengths = [len(tokenizer(s["text"])["input_ids"]) for s in formatted]

print(f"Min tokens:    {min(lengths)}")
print(f"Max tokens:    {max(lengths)}")
print(f"Average tokens: {sum(lengths)/len(lengths):.0f}")
print(f"95th percentile: {sorted(lengths)[int(len(lengths)*0.95)]}")

Min tokens:    330
Max tokens:    1243
Average tokens: 618
95th percentile: 904


In [22]:
from transformers import AutoTokenizer , AutoModelForCausalLM , TrainingArguments
from peft import LoraConfig , prepare_model_for_kbit_training
from trl import SFTTrainer

RuntimeError: Failed to import trl.trainer.sft_trainer because of the following error (look up to see its traceback):
Failed to import transformers.data.data_collator because of the following error (look up to see its traceback):
All ufuncs must have type `numpy.ufunc`. Received (<ufunc 'sph_legendre_p'>, <ufunc 'sph_legendre_p'>, <ufunc 'sph_legendre_p'>)